# 🦠 COVID-19 México — Demo: Pandas · Seaborn

Datos diarios de casos confirmados por estado (Feb 2020 – Jun 2023).  
Fuente: Secretaría de Salud / DGE

In [ ]:
import os
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

DIRECTORY = 'data'
FILE = 'Casos_Diarios_Estado_Nacional_Confirmados_20230625.csv'
PATH = os.path.join(DIRECTORY, FILE)
sns.set_theme(style='darkgrid', palette='tab10')
print('Librerías listas ✔')

---
## 🐼 PARTE 1 — pandas
### 1.1 Carga y vistazo inicial

In [ ]:
datos_crudos = pd.read_csv(PATH)
datos_crudos.head()

### 1.2 Limpieza y transformación

In [ ]:
datos_traspuestos = (datos_crudos.drop(columns=['cve_ent', 'poblacion']).
    set_index('nombre').
    transpose())
datos_traspuestos.index.name = 'fecha'
datos_traspuestos.head()

In [ ]:
datos_formateados = datos_traspuestos.copy().astype(int)
datos_formateados.index = pd.to_datetime(datos_formateados.index, format='%d-%m-%Y')
datos_formateados.dtypes

In [ ]:
datos_formateados.head()

### 1.2.1 Datos nacionales vs. por entidad

In [ ]:
nacional = datos_formateados['Nacional'].to_frame()
nacional.head()

In [ ]:
entidades = datos_formateados.drop(columns=['Nacional'])
entidades.head()

## PARTE 2 — Analítica

### 2.1 Media móvil de 7 días para casos nacionales

In [ ]:
nacional_media_movil_7d = nacional.rolling(window='7d').mean()
nacional_media_movil_7d.head(20)

### 2.2 Variación fines de semana vs. días hábiles

In [ ]:
nacional_dias = nacional.copy()
nacional_dias['dia'] = nacional_dias.index.to_series().dt.day_name()

In [ ]:
dist_dias_nacional = nacional_dias.groupby('dia').mean().round(2).sort_values('Nacional', ascending=False)
dist_dias_nacional

### 2.3 Top 10 Estados con más casos confirmados

In [ ]:
entidades

In [ ]:
top_estados = entidades.sum().sort_values(ascending=False).head(10)
top_estados

2.4 Casos acumulados mensuales por estado

In [ ]:
acumulados_mensuales = entidades.resample('M').sum()
acumulados_mensuales.head()

---
## 🎨 PARTE 3 — Seaborn
### 3.1 Curva nacional con media móvil

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

sns.lineplot(data=nacional,
    x=nacional.index, 
    y='Nacional', 
    color='steelblue', 
    linewidth=1.5, 
    label='Casos diarios', 
    ax=ax)
sns.lineplot(data=nacional_media_movil_7d,
    x=nacional_media_movil_7d.index, 
    y='Nacional',
    color='green',
    linewidth=2.5,
    label='Media móvil 7d',
    ax=ax)
ax.set_title('Curva epidémica nacional de COVID-19 en México', fontsize=16)
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Número de casos', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('fig_curva_nacional.png', dpi=150)
plt.show()

### 3.2 Boxplot por día de la semana

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))
ax1 = fig.add_subplot(2, 1, 1)
sns.barplot(data=nacional_dias,
    x='dia',
    y='Nacional',
    order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
    palette='tab10',
    ax=ax1)
ax1.set_title('Promedio de casos por día de la semana', fontsize=14)
ax1.set_xlabel('Día de la semana', fontsize=12)
ax1.set_ylabel('Número promedio de casos', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax2 = fig.add_subplot(2, 1, 2)
ax2.pie(dist_dias_nacional['Nacional'],
    labels=dist_dias_nacional.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('tab10', n_colors=7))
ax2.set_title('Distribución porcentual por día de la semana', fontsize=14)
plt.tight_layout()

### 3.2 Barplot — top 10 estados

In [ ]:
top_estados.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 10 Estados con más casos confirmados de COVID-19', fontsize=16)
plt.xlabel('Número total de casos (millones)', fontsize=12)
plt.ylabel('Estado', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()

### 3.3 Heatmap mensual — top 10 estados

In [ ]:
meses = acumulados_mensuales.index.strftime('%Y-%m')
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(acumulados_mensuales[top_estados.index].T,
    cmap='Reds',
    linewidths=0.5,
    linecolor='gray',
    cbar_kws={'label': 'Número de casos'})
ax.set_title('Calor mensual de casos confirmados por estado (Top 10)', fontsize=16)
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Estado', fontsize=12)
ax.set_xticklabels(meses, rotation=45, ha='right', fontsize=8)
plt.tight_layout()